In [ ]:
import itertools
import pandas as pd

import part1

In [ ]:
MODELS = {
    "mlp_indicator-lemmas": "results/mlp_indicator-lemmas/cv_scores.csv",
    "mlp_bow-lemmas": "results/mlp_bow-lemmas/cv_scores.csv",
    "mlp_embeddings-sentence": "results/mlp_embeddings/cv_scores.csv",
    "logistic_indicator-lemmas": "results/logistic_indicator-lemmas/cv_scores.csv",
    "logistic_bow-lemmas": "results/logistic_bow-lemmas/cv_scores.csv",
    "logistic_embeddings-sentence": "results/logistic_embeddings/cv_scores.csv",
    # add cnn or two hidden layer MLPs here
}

METRIC = "f1"    # or "accuracy"
SEED = 0
ROPE = 0.005

In [ ]:
all_scores = {name: pd.read_csv(path) for name, path in MODELS.items()}

# getting all measurements for a model
def get_all_measurements(model_name):
    df = all_scores[model_name]
    df = df[df["seed"] == SEED].sort_values("fold")
    return df[METRIC].values

In [ ]:
def compare_all_pairs(rope_size):
    rows = []
    for a, b in itertools.combinations(MODELS, 2):
        posterior = part1.analysis.BenavoliCorrelatedT(
            get_all_measurements(a),
            get_all_measurements(b),
            names=[a, b],
        )
        p_a, p_tie, p_b = posterior.probs(rope_size)
        rows.append([a, b, p_a, p_tie, p_b])
    return pd.DataFrame(rows, columns=["A", "B", "P(A wins)", "P(Tie)", "P(B wins)"])

compare_all_pairs(ROPE).round(4)

In [ ]:
# checking how sensitive the results are to the ROPE size
for rope in [0.0025, 0.005, 0.01]:
    print(f"ROPE = {rope}")
    display(compare_all_pairs(rope).round(4))